# 🔧 Day 5 — Training Stability + Debugging
**Vision System Capstone v3** | Gradient Monitor · NaN Detection · Overfit Test · Loss Curves

This notebook covers:
1. Build model + dataloaders (compatible with 80 classes)
2. Weight initialisation check
3. Overfit test — 128 samples, 10 epochs
4. Gradient norm monitoring
5. NaN/Inf detection demo
6. Loss curves from training log
7. Day 5 summary

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {device}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Load configs ──────────────────────────────────────────────────────────────
with open('../configs/data_config.yaml')  as f: DCFG = yaml.safe_load(f)
with open('../configs/model_config.yaml') as f: MCFG = yaml.safe_load(f)
with open('../configs/train_config.yaml') as f: TCFG = yaml.safe_load(f)

NUM_CLASSES = DCFG['dataset']['num_classes']
print(f'Dataset    : {DCFG["dataset"]["name"]}')
print(f'Num classes: {NUM_CLASSES}')
print(f'AMP enabled: {TCFG["amp"]["enabled"]}')

---
## 1. Build Model + DataLoaders

In [ ]:
from datasets.dataset_loader import get_dataloaders, load_config as load_data_cfg
from models.resnet import build_model
from datasets.augmentation import SoftCrossEntropyLoss

data_cfg = load_data_cfg('../configs/data_config.yaml')
loaders  = get_dataloaders(data_cfg)

model     = build_model(
    model_cfg_path='../configs/model_config.yaml',
    data_cfg_path ='../configs/data_config.yaml',
).to(device)
criterion = SoftCrossEntropyLoss(smoothing=TCFG['loss']['label_smoothing'])

print(f'Train batches: {len(loaders["train"])}')
print(f'Model params : {model.count_params():,}')
print(f'Output shape : ({TCFG["dataloader"]["batch_size"] if "batch_size" in str(TCFG) else 64}, {NUM_CLASSES})')

---
## 2. Weight Initialisation Check

In [ ]:
from training.stability import check_weight_init

results = check_weight_init(model)

print('\nWeight Init Results:')
print(f'  Conv std mean    : {results["conv_std_mean"]:.4f} (Kaiming He -- varies by layer size)')
print(f'  BN gamma mean    : {results["bn_gamma_mean"]:.4f} (expected ~1.0) {"OK" if results["bn_gamma_ok"] else "FAIL"}')
print(f'  BN beta mean     : {results["bn_beta_mean"]:.4f} (expected ~0.0) {"OK" if results["bn_beta_ok"] else "FAIL"}')
print(f'  Zero-init blocks : {results["zero_init_count"]} {"OK" if results["zero_init_ok"] else "FAIL"}')
print(f'  All passed       : {results["all_passed"]}')

if results['all_passed']:
    print('\n  Weights correctly initialised -- safe to train')
else:
    print('\n  Check zero_init_residual in configs/model_config.yaml')

---
## 3. Overfit Test -- 128 Samples, 10 Epochs

In [ ]:
from training.stability import run_overfit_test

print('Running overfit test...')
print('Expected: acc >= 0.99 within 10 epochs')
print('If fails: check data pipeline or model capacity\n')

ov_results = run_overfit_test(
    model       = model,
    loader      = loaders['train'],
    criterion   = criterion,
    device      = device,
    num_classes = NUM_CLASSES,
    n_samples   = 128,
    max_epochs  = 10,
    amp_enabled = TCFG['amp']['enabled'],
)

print(f'\nResult: {"PASSED" if ov_results["passed"] else "FAILED"}')
print(f'Final acc  : {ov_results["final_acc"]:.4f}')
print(f'Final loss : {ov_results["final_loss"]:.4f}')
print(f'Epochs taken: {ov_results["epochs_taken"]}')

In [ ]:
# Plot overfit test curve
if ov_results['history']:
    epochs = [h['epoch'] for h in ov_results['history']]
    accs   = [h['acc']   for h in ov_results['history']]
    losses = [h['loss']  for h in ov_results['history']]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, losses, color='#e74c3c', linewidth=2, marker='o', markersize=4)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Overfit Test -- Loss'); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, accs, color='#2ecc71', linewidth=2, marker='o', markersize=4)
    axes[1].axhline(0.99, color='black', linestyle='--', linewidth=1, label='Target: 0.99')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Overfit Test -- Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

    status = 'PASSED' if ov_results['passed'] else 'FAILED'
    plt.suptitle(f'Overfit Test (128 samples) -- {status}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    Path('../logs').mkdir(exist_ok=True)
    plt.savefig('../logs/overfit_test.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved --> logs/overfit_test.png')

---
## 4. Gradient Norm Monitoring

In [ ]:
from training.monitor import GradientMonitor
from datasets.augmentation import AugmentationManager
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F

monitor     = GradientMonitor(model, log_every_n=1)
aug_manager = AugmentationManager.from_config(data_cfg)
optimizer   = torch.optim.AdamW(model.parameters(), lr=TCFG['optimizer']['lr'])
scaler      = GradScaler(enabled=TCFG['amp']['enabled'])
amp_on      = TCFG['amp']['enabled']

# Run 5 batches to collect gradient norms
model.train()
grad_norms_history = []

print('Running 5 batches to measure gradient norms...')
for batch_idx, (images, labels) in enumerate(loaders['train']):
    if batch_idx >= 5:
        break

    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    images, soft_labels = aug_manager.apply(images, labels)

    optimizer.zero_grad()
    with autocast(enabled=amp_on):
        logits = model(images)
        loss   = criterion(logits, soft_labels)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)

    norms = monitor.record(batch_idx)
    grad_norms_history.append(norms)

    scaler.step(optimizer)
    scaler.update()
    print(f'  Batch {batch_idx}: loss={loss.item():.4f}')

summary = monitor.summarize(epoch=0)
print(f'\nGradient norm summary:')
for group, norm in summary.items():
    status = ''
    if norm > 10.0:  status = ' EXPLODING'
    elif norm < 1e-4: status = ' VANISHING'
    else:             status = ' OK'
    print(f'  {group:<12}: {norm:.6f}{status}')

In [ ]:
# Plot gradient norms per layer
if summary:
    fig, ax = plt.subplots(figsize=(10, 4))
    groups = list(summary.keys())
    norms  = list(summary.values())
    colors = ['#e74c3c' if n > 10 else '#f39c12' if n < 1e-4 else '#2ecc71' for n in norms]

    bars = ax.bar(groups, norms, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(10.0, color='red',   linestyle='--', linewidth=1, label='Explode threshold (10.0)')
    ax.axhline(1e-4, color='orange',linestyle='--', linewidth=1, label='Vanish threshold (1e-4)')

    for bar, val in zip(bars, norms):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Layer Group'); ax.set_ylabel('Mean Gradient Norm')
    ax.set_title('Gradient Norms per Layer -- Healthy = all green')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../logs/gradient_norms.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved --> logs/gradient_norms.png')

---
## 5. NaN/Inf Detection Demo

In [ ]:
from training.stability import check_nan_inf

# Case 1: Normal logits (should pass)
model.eval()
with torch.no_grad():
    dummy  = torch.randn(4, 3, 224, 224).to(device)
    logits = model(dummy)
    soft_l = F.one_hot(torch.zeros(4, dtype=torch.long).to(device), NUM_CLASSES).float()
    loss   = criterion(logits, soft_l)

result = check_nan_inf(logits, loss, batch_idx=0, epoch=1)
print('Case 1 -- Normal logits:')
print(f'  has_nan    : {result["has_nan"]}')
print(f'  has_inf    : {result["has_inf"]}')
print(f'  logit range: [{result["logit_min"]:.3f}, {result["logit_max"]:.3f}]')
print(f'  loss_val   : {result["loss_val"]:.4f}')
print(f'  Status     : {"CLEAN" if not result["has_nan"] and not result["has_inf"] else "PROBLEM DETECTED"}')

# Case 2: Simulated NaN (shows what bug looks like)
print('\nCase 2 -- Simulated NaN (debug_log Bug #2):')
nan_logits = torch.full((4, NUM_CLASSES), float('nan')).to(device)
nan_loss   = torch.tensor(float('nan')).to(device)
result2    = check_nan_inf(nan_logits, nan_loss, batch_idx=47, epoch=3)
print(f'  has_nan : {result2["has_nan"]} (expected True)')
print(f'  Fix     : LR finder -> reduce LR 10x -> restart training')

---
## 6. Loss Curves from Training Log

In [ ]:
from training.stability import save_loss_curves

log_path = Path('../logs/training_log.csv')

if not log_path.exists():
    print('training_log.csv not found yet.')
    print('Run training first: python training/train.py')
    print('Then re-run this cell.')
else:
    import pandas as pd
    df = pd.read_csv(log_path)
    print(f'Epochs logged: {len(df)}')
    print(df.tail(3).to_string(index=False))

    saved = save_loss_curves(
        csv_path  = str(log_path),
        save_path = '../logs/loss_curves/training_curves.png'
    )

    if saved:
        from PIL import Image as PILImage
        img = PILImage.open('../logs/loss_curves/training_curves.png')
        plt.figure(figsize=(16, 5))
        plt.imshow(img); plt.axis('off')
        plt.tight_layout(); plt.show()

    # Overfit check
    if len(df) >= 5:
        gap = (df.tail(5)['train_acc'] - df.tail(5)['val_acc']).mean()
        print(f'\nOverfit check (last 5 epochs): gap = {gap:.4f}')
        if gap > 0.15:
            print('  Overfitting -- add more augmentation or increase dropout')
        elif gap > 0.08:
            print('  Mild overfit -- monitor closely')
        else:
            print('  Healthy -- gap is acceptable')

---
## 7. Stability Tracker Demo

In [ ]:
from training.stability import StabilityTracker

# Demo: simulate 5 epochs of data
tracker = StabilityTracker()

simulated = [
    (1, 2.8, 2.6, 0), (2, 2.1, 2.0, 0),
    (3, 1.6, 1.7, 0), (4, 1.2, 1.4, 0),
    (5, 0.9, 1.1, 0),
]
for epoch, tl, vl, nans in simulated:
    tracker.record_epoch(epoch, tl, vl, nan_count=nans)

report = tracker.report()
print(f'\nStability report:')
print(f'  Stable       : {report["stable"]}')
print(f'  NaN events   : {report["total_nans"]}')
print(f'  Best val loss: {report["best_val"]}')
print(f'  Final gap    : {report["final_gap"]}')

In [ ]:
# Day 5 Summary
print('=' * 60)
print('DAY 5 SUMMARY')
print('=' * 60)
print(f'  Dataset      : {DCFG["dataset"]["name"]} ({NUM_CLASSES} classes)')
print(f'  Model        : ResNet-{MCFG["arch"]["depth"]} + SE')
print()
print('  CHECKS:')
print(f'    Weight init  : {"PASSED" if results["all_passed"] else "FAILED"}')
print(f'    Overfit test : {"PASSED" if ov_results["passed"] else "FAILED"} (acc={ov_results["final_acc"]:.3f})')
print(f'    Grad norms   : {"OK" if all(v < 10 for v in summary.values()) else "CHECK REQUIRED"}')
print(f'    NaN detect   : CLEAN (normal forward pass)')
print()
print('  PLOTS SAVED:')
for f in ['logs/overfit_test.png', 'logs/gradient_norms.png', 'logs/loss_curves/training_curves.png']:
    exists = 'YES' if Path(f'../{f}').exists() else 'run training first'
    print(f'    {f}: {exists}')
print()
print('  INTERVIEW:')
print('    "I ran an overfit test before full training -- 128 samples')
print('     in 10 epochs. If model cannot overfit, the data pipeline')
print('     has a bug. I tracked gradient norms every batch -- NaN')
print('     at epoch 3 was caught by check_nan_inf, traced to AMP')
print('     + high LR, fixed by LR finder. Debug log has 8 entries."')